In [ ]:

# Install Dependencies
!pip install transformers sentencepiece sentence-transformers faiss-cpu gradio langdetect


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
import faiss, numpy as np
from langdetect import detect

# Multilingual embeddings
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# Multilingual LLM
tokenizer = AutoTokenizer.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")




docs = [
    "MPF is Hong Kong’s mandatory pension scheme that helps residents save for retirement.",
    "Insurance policies protect families against financial risks such as accidents or illness.",
    "Investments carry risk; diversification across asset classes is key to reducing exposure.",
    "Budgeting helps track income and expenses, ensuring financial stability.",
    "Emergency funds provide a safety net for unexpected expenses.",

    "MPF 是香港的強制性退休計劃，幫助居民為退休儲蓄。",
    "保險可以保護家庭免受意外或疾病帶來的財務風險。",
    "投資有風險；跨資產類別的多元化是降低風險的關鍵。",
    "制定預算有助於跟踪收入和支出，確保財務穩定。",
    "緊急基金為突發支出提供安全網。"
]


doc_embeddings = embedder.encode(docs)
index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(np.array(doc_embeddings))


In [37]:
# Multilingual Chat Interface

def ask_bot(query):
    # Detect input language
    lang = detect(query)

    # Retrieve docs
    q_emb = embedder.encode([query])
    D, I = index.search(np.array(q_emb), k=2)
    contexts = [docs[i] for i in I[0]]

    # Build prompt with explicit language instruction
    if lang.startswith("zh"):
        prompt = f"請用中文回答。\n上下文: {' '.join(contexts)}\n問題: {query}\n答案:"
    else:
        prompt = f"Answer in English.\nContext: {' '.join(contexts)}\nQuestion: {query}\nAnswer:"

    # Generate answer
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_length=128)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer, "\n".join(contexts)


In [ ]:
# Gradio Interface

with gr.Blocks() as demo:
    gr.Markdown("## 💬 Financial Literacy Chatbot (Hong Kong)")

    query = gr.Textbox(label="Ask a question (English, Cantonese, Mandarin)")
    answer = gr.Textbox(label="Answer")
    source = gr.Textbox(label="Source Documents")

    btn = gr.Button("Ask")
    btn.click(fn=ask_bot, inputs=query, outputs=[answer, source])

demo.launch()
